# Catalog SQL demo — consumindo as tabelas por nome

Pré-requisito (rode no terminal antes):
```bash
make demo                                   # gera dados + bronze/silver/gold
python -m ifood_case.main --stage catalog   # registra no metastore
```

Depois desse notebook abrir, você consulta as tabelas Delta como num data warehouse de verdade.

In [ ]:
from ifood_case.spark import build_spark
from ifood_case.config import load_config
from ifood_case import catalog

cfg = load_config()
spark = build_spark(delta=cfg.storage_format == 'delta')
spark.sparkContext.setLogLevel('WARN')

# Registra Bronze/Silver/Gold no metastore (idempotente): cria o schema `ifood`
# e as tabelas externas, para os SELECTs abaixo rodarem sem pre-requisito.
registered = catalog.run(spark, cfg)
print('database:', cfg.database, '| warehouse:', cfg.warehouse_dir)
print('tabelas registradas:', list(registered))

## 1. O catalog tem mesmo as tabelas registradas?

In [ ]:
spark.sql('SHOW DATABASES').show()
spark.sql(f'SHOW TABLES IN {cfg.database}').show(truncate=False)

## 2. Q1 — receita média por mês (consulta SQL pura, sem path)

In [ ]:
spark.sql(f'''
    SELECT trip_month AS mes,
           COUNT(*) AS qtd_corridas,
           ROUND(AVG(total_amount), 2) AS receita_media_usd,
           ROUND(SUM(total_amount), 2) AS receita_total_usd
    FROM {cfg.database}.silver_trips
    GROUP BY trip_month
    ORDER BY trip_month
''').show()

## 3. Q2 — passageiros por hora em maio

In [ ]:
spark.sql(f'''
    SELECT HOUR(tpep_pickup_datetime) AS hora,
           COUNT(*) AS qtd_corridas,
           ROUND(AVG(passenger_count), 3) AS media_pax
    FROM {cfg.database}.silver_trips
    WHERE trip_month = 5
    GROUP BY HOUR(tpep_pickup_datetime)
    ORDER BY hora
''').show(24)

## 4. Metadados — descrição da tabela (mostra LOCATION, partições, schema)

In [ ]:
spark.sql(f'DESCRIBE EXTENDED {cfg.database}.silver_trips').show(50, truncate=False)

## 5. Time Travel — DESCRIBE HISTORY (só funciona em Delta)

In [ ]:
if cfg.storage_format == 'delta':
    spark.sql(f'DESCRIBE HISTORY {cfg.database}.silver_trips') \
        .select('version', 'timestamp', 'operation', 'operationMetrics') \
        .show(truncate=False)
else:
    print('storage_format != delta -> Time Travel indisponível.')

## 6. JOIN entre tabelas catalogadas (Bronze x Silver)

In [ ]:
spark.sql(f'''
    SELECT 'bronze' AS camada, COUNT(*) AS linhas FROM {cfg.database}.bronze_trips
    UNION ALL
    SELECT 'silver',           COUNT(*)          FROM {cfg.database}.silver_trips
    UNION ALL
    SELECT 'gold_trips',       COUNT(*)          FROM {cfg.database}.gold_trips
''').show()